# Lab 01 - Histograms manipulations and Filtering

Computer Vision 2024-25 (M. Caligiuri, P. Zanuttigh, F. Barbato and F. Lincetto)

Student: GIACOMO CALABRIA - 2122530

In this lab you will explore two different topics:

1. Histogram-based processing:
    - Histogram equalization
    - Histogram matching.
2. Filtering for denoising:
    - Gaussian filter
    - Median filter
    - Bilateral filter.

---

## Import the necessary libraries

In [ ]:
# Import the necessary packages
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt

---

## Histogram equalization

In this section you will have to perform the histogram equalization of the image `./data/castle.jpg`. Do that using first the **BGR** color space and then the **LAB** color space. You will see the difference in the results.

### Auxiliary functions

Write here all the auxiliary functions you will need.
More precisely you have to implement the following functions:
- `calc_hist` -> given in input an image it will return a list containing the histogram of each channel,
- `show_histogram` -> given in input the list of all the histograms of an image it will return an image containing the plot of the histograms of each channel side by side (it uses matplotlib to generate the plots then convert them to images using `cv2`),

  ![hists_example](data/hists.png)
- `img_hist` -> given in input an image and the correspondent list of histograms it will return the image with the histograms plotted on top of it (call `show_histogram`).

  ![img_hist_example](data/img_hists.png)

In [ ]:
# Write a function to compute the histogram of the image (channel by channel)
def calc_hists(img: np.ndarray) -> list:
    """
    Calculates the histogram of the image (channel by channel).

    Args:
        img (numpy.ndarray): image to calculate the histogram
    
    Returns:
        list: list of histograms
    """
    # Check if the image is grayscale or color
    if(len(img.shape) == 2):
        #Compute the histogram of the image
        hist = cv.calcHist([img],[0],None,[256],[0,256], accumulate=False)

        return hist
    
    else:
        # Create a empty list to store the histograms
        hist = [0, 0, 0]

        #Compute the histogram of the image for each channel
        for i in range(img.shape[2]):
            hist[i] = cv.calcHist([img],[i],None,[256],[0,256], accumulate=False)
        
        return hist

In [ ]:
# Write a function to plot together the three histograms of the image
# (plot only one if it is grayscale)
def show_histogram(hists: list) -> np.ndarray:
    """
    Shows the histogram of the image.
    
    Args:
        hists (list): list of histograms to plot

    Returns:
        img (numpy.ndarray): image containing the histograms
     """   
    if(len(hists) == 1):

        # Create a figure to plot the histogram
        fig = plt.figure()
        plt.plot(hists[0])
        plt.title("Gray Histogram")
        plt.xlabel("Bins")
        plt.ylabel("# of Pixels")
        plt.xlim([0, 256])
        #plt.show()

        # Usa la canvas per ottenere l'immagine dalla figura matplotlib
        fig.canvas.draw()
        
        # Ottieni l'immagine dalla canvas come un array RGBA
        hist_img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        hist_img = hist_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        
        plt.close()  # Chiude la figura per liberare memoria
        
        # Convert image to cv2 format
        return cv.cvtColor(hist_img, cv.COLOR_RGB2BGR)
        
    else: 
        colors = ['blue', 'green', 'red']
        titles = ['Blu Histogram', 'Green Histogram', 'Red Histogram']

        fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

        for i, ax in enumerate(axes):
            ax.plot(hists[i], color=colors[i])
            ax.set_title(titles[i])
            ax.set_xlabel("Bins")
            ax.set_ylabel("# of Pixels")
            ax.set_xlim([0, 256])
            
        plt.tight_layout()
        #plt.show()

        # Usa la canvas per ottenere l'immagine dalla figura matplotlib
        fig.canvas.draw()
        
        # Ottieni l'immagine dalla canvas come un array RGBA
        hist_img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        hist_img = hist_img.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        
        plt.close(fig)  # Chiude la figura per liberare memoria
        
        # Convert image to cv2 format
        return cv.cvtColor(hist_img, cv.COLOR_RGB2BGR)

In [ ]:
# Write a function to fuse the histogram image with the original one
def img_hist(img: np.ndarray, hists: list[np.ndarray], scaling_factor: float = 0.3) -> np.ndarray:
    """
    Fuse the image with the histogram.
    
    Args:
        img (np.ndarray): image
        hist (list(np.ndarray)): histograms of the image
        
    Returns:
        img (numpy.ndarray): image containing the original image with the histograms on top
    """
    
    # Get the histogram image
    hist_img = show_histogram(hists)

    # Check if the image is grayscale, if so convert it to BGR
    if len(img.shape) == 2:
        img = cv.cvtColor(img, cv.COLOR_GRAY2BGR)
    
    # Resize the histogram image
    # Calcola il rapporto d'aspetto per l'immagine degli istogrammi
    aspect_ratio = hist_img.shape[0] / hist_img.shape[1]

    # Nuova altezza per mantenere il rapporto d'aspetto
    new_hist_height = int(img.shape[1] * aspect_ratio)

    # Ridimensiona l'immagine degli istogrammi
    hist_img = cv.resize(hist_img, (img.shape[1], new_hist_height))

    new_height = img.shape[0] + hist_img.shape[0]
    combined_img = np.zeros((new_height, img.shape[1], 3), dtype=np.uint8)

    # Incolla l'immagine degli istogrammi e l'immagine originale
    combined_img[:hist_img.shape[0], :, :] = hist_img
    combined_img[hist_img.shape[0]:, :, :] = img
    
    return combined_img


In [ ]:
# Load one image and show it
image = cv.imread('data/barbecue.jpg')

# Convert to grayscale (optional)
#image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)

# Compute the histograms
hist_list = calc_hists(image)

# Fuse the histogram image with the original one
out_image = img_hist(image, hist_list)

# Show the image with the histograms
cv.namedWindow('Histogram', cv.WINDOW_GUI_NORMAL)
cv.imshow('Histogram', out_image)
key = cv.waitKey(0)
cv.destroyAllWindows()

### Equalize the image in the BGR space

Equalize the image `./data/castle.jpg` (or if you prefer `./data/sea.jpg`) in the BGR space and show the result.

In [ ]:
# Load image and and compute its histogram

# Add your code here
image = cv.imread('data/portello.png')

# Convert to grayscale (optional)
#image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)

# Compute the histogram of the image
hist_list = calc_hists(image)

In [ ]:
# Show the image with its histogram before the equalization

# Add your code here

# Fuse the histogram image with the original one
beforeeq = img_hist(image, hist_list)

# Show the image with the histograms
cv.namedWindow('Before Equalization', cv.WINDOW_GUI_NORMAL)
cv.imshow('Before Equalization', beforeeq)
key = cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
# Equalize the histogram of the image

# Add your code here
def equalize_image(image: np.ndarray) -> np.ndarray:
    # Check if the image is grayscale or color
    if len(image.shape) == 2:
        # Equalize the histogram of the image
        equalized_image = cv.equalizeHist(image) 
        
        return equalized_image
    else:
        # Allocate memory for the equalized image
        equalized_image = np.zeros(image.shape, dtype=np.uint8)

        # Equalize the histogram of the image for each channel
        equalized_image[:, :, 0] = cv.equalizeHist(image[:, :, 0])
        equalized_image[:, :, 1] = cv.equalizeHist(image[:, :, 1])
        equalized_image[:, :, 2] = cv.equalizeHist(image[:, :, 2])
        
        return equalized_image


In [ ]:
# Show the image and the relative histogram (in the same window)

# Add your code here
img_eq = equalize_image(image)

aftereq = img_hist(img_eq, calc_hists(img_eq))
cv.namedWindow('After Equalization', cv.WINDOW_GUI_NORMAL)
cv.imshow('After Equalization', aftereq)
key = cv.waitKey(0)
cv.destroyAllWindows()

### Equalize the image in the LAB space

Equalize the image (`./data/castle.jpg` or `./data/sea.jpg`) in the LAB space and show the result.

In [ ]:
# Load the image and convert it to LAB colorspace

# Add your code here
image = cv.imread('data/portello.png')

# Convert the image to the LAB color space
lab_image = cv.cvtColor(image, cv.COLOR_BGR2LAB)

# Equalize the histogram of the L channel

# Add your code here
lab_image[:, :, 0] = cv.equalizeHist(lab_image[:, :, 0])

# Convert the image back to BGR colorspace

# Add your code here
equalized_image = cv.cvtColor(lab_image, cv.COLOR_LAB2BGR)

# Extract the histograms for each channel

# Add your code here
hist_list = calc_hists(equalized_image)

In [ ]:
# Show the image and its relative histogram

# Add your code here
aftereq = img_hist(equalized_image, hist_list)

cv.namedWindow('After Equalization', cv.WINDOW_GUI_NORMAL)
cv.imshow('After Equalization', aftereq)
key = cv.waitKey(0)
cv.destroyAllWindows()

---

## Histogram matching

In this section you will have to perform the histogram matching of the image `./data/panorama_left.jpg` to the image `./data/panorama_right.jpg`. Do that in the **BGR** color space.

### Histogram matching function

The function to perform the histogram matching is provided here. The function takes in input the image to be matched and the image to match and will return the matched image.

In [ ]:
#  function to perform the histogram matching
def hist_match(source: np.ndarray, reference: np.ndarray) -> np.ndarray:
    """
    Adjust the pixel values of a color image such that its histogram
    matches that of a target one.

    Args:
        source (numpy.ndarray): Image to transform; the histogram is computed over the flattened array
        reference (numpy.ndarray): Template image; can have different dimensions from source
    Returns:
        numpy.ndarray: The transformed output image
    """
    
     # Assert that the images have the same number of channels (grayscale or RGB) and the same dimensions
    assert source.shape[2] == reference.shape[2], "Images must have the same number of channels"
    assert source.shape[:2] == reference.shape[:2], "Images must have the same dimensions"

    # Compute the source image's histogram and CDF
    src_hists = [np.histogram(source[..., i].flatten(), 256, [0,256])[0] for i in range(source.shape[2])]
    src_cdfs = [hist.cumsum() for hist in src_hists]
    src_cdfs_normalized = [cdf / float(cdf.max()) for cdf in src_cdfs]
 
    # Compute the reference image's histogram and CDF
    ref_hists = [np.histogram(reference[..., i].flatten(), 256, [0,256])[0] for i in range(reference.shape[2])]
    ref_cdfs = [hist.cumsum() for hist in ref_hists]
    ref_cdfs_normalized = [cdf / float(cdf.max()) for cdf in ref_cdfs]

    # Create a lookup table to map pixel values from the source to the reference
    lookup_tables = [np.zeros(256) for _ in range(source.shape[2])]
    lookup_values = [0] * source.shape[2]
    for index in range(len(lookup_tables)):
        for src_pixel_val in range(len(src_cdfs_normalized[index])):
            lookup_values[index]
            for ref_pixel_val in range(len(ref_cdfs_normalized[index])):
                if ref_cdfs_normalized[index][ref_pixel_val] >= src_cdfs_normalized[index][src_pixel_val]:
                    lookup_values[index] = ref_pixel_val
                    break
            lookup_tables[index][src_pixel_val] = lookup_values[index]

    # Apply the lookup table to the source image
    matched = np.stack([cv.LUT(source[..., i], lookup_tables[i]).astype(np.uint8) for i in range(len(lookup_tables))], axis=-1)

    return matched

### Match the images

In [ ]:
# Load the two iamges

# Add your code here
image1 = cv.imread('data/panorama_left.jpg')
image2 = cv.imread('data/panorama_right.jpg')

# Convert the images to BGR colorspace

# They are already in BGR colorspace so no need to convert them. 

In [ ]:
# Visualize the two images with the relative histograms before the histogram matching
# Visualize the two images side by side in the same widow

# Add your code here
# Compute the histograms
hist_list1 = calc_hists(image1)
hist_list2 = calc_hists(image2)

# Fuse the histogram image with the original one
before1 = img_hist(image1, hist_list1)
before2 = img_hist(image2, hist_list2)

# Show the image with the histograms
cv.namedWindow('Before Matching', cv.WINDOW_GUI_NORMAL)
cv.imshow('Before Matching', np.hstack([before1, before2]))
key = cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
# Match the histograms of the two images

# Add your code here

# Perform the histogram matching
matched = hist_match(image1, image2)

# Visualize the two iamges with the relative histograms
# Visualize the two images side by side in the same widow

# Add your code here
# Compute the histograms
hist_list1 = calc_hists(matched)
hist_list2 = calc_hists(image2)

# Fuse the histogram image with the original one
after1 = img_hist(matched, hist_list1)
after2 = img_hist(image2, hist_list2)

# Show the image with the histograms
cv.namedWindow('After Matching', cv.WINDOW_GUI_NORMAL)
cv.imshow('After Matching', np.hstack([after1, after2]))
key = cv.waitKey(0)
cv.destroyAllWindows()

---

## Image denoising

In this section you will have to perform the denoising of the image `./data/overexposed.jpg` . Do that using the following filters:

    Gaussian filter,
    Median filter,
    Bilateral filter.

For each filter you will have to try different values for the parameters in order to see the effect of each parameter on the result. To do that in an interactive way it is required that you implement the cv.trackbar function. In this way you will be able to change the parameters of the filters and to see the result in real time.

In order to build a better code you have to implement each filter as a class (child of a main filter class called Filter). The structure to do this is already provided in this notebook

### Define the filters classes

In [ ]:
# Define the main Filter class with the common methods
# (this class will be used as a base class for the other filters)
class Filter:
    def __init__(self, size: int):
        # Constructor of the class
        if size % 2 == 0:
            size += 1
        self.set_filter_size(size)
    
    def set_filter_size(self, size: int) -> None:
        # Method to set the filter size
        if size % 2 == 0:
            size += 1
        self.filter_size = size
    
    def get_filter_size(self) -> int:
        # Method to get the filter size
        return self.filter_size

    def __call__(self, image: np.ndarray) -> None:
        # Method to perform the filtering
        # (it must be implemented in the derived classes)
        raise NotImplementedError

In [ ]:
# Define the class for the gaussian filter
class GaussianFilter(Filter):
    def __init__(self, size: int, sigma_g: float):
        super().__init__(size)
        self.sigma = sigma_g

    def set_sigma(self, s: float) -> None:
        # Method to set the sigma value

        # Add your code here (remove the pass statement)
        if not isinstance(s, (float, int)):
            raise ValueError("Sigma must be a numeric value.")

        if s <= 0:
            raise ValueError("Sigma must be greater than 0.")
        
        self.sigma = s


    def get_sigma(self) -> float:
        # Method to get the sigma value
        
        # Add your code here (remove the pass statement)
        if self.sigma is None:
            raise ValueError("Sigma has not been set.")
           
        return self.sigma
    
    def __call__(self, image: np.ndarray, size: int, sigma: float) -> None:
        # Method to call the filter (it is called when the object is called)
        # This function will be called by the trackbar object and will update the imamge to show
        
        # Add your code here (remove the pass statement)
        self.set_sigma(sigma)
        self.set_filter_size(size)
    
        out_image = np.copy(image)
        out_image = cv.GaussianBlur(image,(self.get_filter_size(),self.get_filter_size()),self.get_sigma(),0,cv.BORDER_DEFAULT)
        cv.imshow('Gaussian Filter',out_image)


# Define the class for the median filter
class MedianFilter(Filter):
    def __init__(self, size: int):
        super().__init__(size)

    def __call__(self, image: np.ndarray, size: int) -> None:
        # Method to call the filter (it is called when the object is called)
        # This function will be called by the trackbar object and will update the image to show
        
        # Add your code here (remove the pass statement)
        self.set_filter_size(size)
        out_image = np.copy(image)
        out_image = cv.medianBlur(image, self.get_filter_size())
        cv.imshow('Median Filter',out_image)


# Define the class for the bilateral filter
class BilateralFilter(Filter):
    def __init__(self, size: int, sigma_s: float, sigma_r: float):
        super().__init__(size)
        self.sigma_space = sigma_s
        self.sigma_range = sigma_r

    def set_sigma_range(self, sr: float) -> None:
        # Method to set the sigma range value

        # Add your code here (remove the pass statement)
        if not isinstance(sr, (float, int)):
            raise ValueError("Sigma range must be a numeric value.")

        if sr <= 0:
            raise ValueError("Sigma range must be greater than 0.")

        self.sigma_range = float(sr)

    def get_sigma_range(self) -> float:
        # Method to get the sigma range value
        
        # Add your code here (remove the pass statement)
        if self.sigma_range is None:
            raise ValueError("Sigma range has not been set.")
        return self.sigma_range

    def set_sigma_space(self, ss: float) -> None:
        # Method to set the sigma space value

        # Add your code here (remove the pass statement)
        if not isinstance(ss, (float, int)):
            raise ValueError("Sigma space must be a numeric value.")

        if ss <= 0:
            raise ValueError("Sigma space must be greater than 0.")

        self.sigma_space = float(ss)

    def get_sigma_space(self) -> float:
        # Method to get the sigma space value
        
        # Add your code here (remove the pass statement)
        if self.sigma_space is None:
            raise ValueError("Sigma space has not been set.")
        return self.sigma_space
    
    def __call__(self, image: np.ndarray, sigma_s: float, sigma_r: float) -> None:
        # Method to call the filter (it is called when the object is called)
        # This function will be called by the trackbar object and will update the imamge to show
        
        # Add your code here (remove the pass statement)
        self.set_sigma_space(sigma_s)
        self.set_sigma_range(sigma_r)

        out_image = np.copy(image)
        out_image = cv.bilateralFilter(image,self.get_filter_size(),self.get_sigma_space(),self.get_sigma_range())
        cv.imshow('Bilateral Filter',out_image)

### Load the image and initialize the filters objects

In [ ]:
# Load and show the image to filter

# Add your code here
image = cv.imread('data/overexposed.jpg')

cv.namedWindow('Original Image', cv.WINDOW_GUI_NORMAL)
cv.imshow('Original Image', image)
key = cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
# Initialize the filters

# Add your code here
median = MedianFilter(5)
gaussian = GaussianFilter(5, 1)
bilateral = BilateralFilter(5, 1, 1)


### Test the filters

In [ ]:
# MEDIAN BLUR FILTER

# Create a window for the trackbars

# Add your code here
cv.namedWindow('Median Filter', cv.WINDOW_GUI_NORMAL)

# Create the trackbar

# Add your code here
cv.createTrackbar("Kernel Size = ", "Median Filter", 0, 100, lambda S: median(image,S))

# Show the filtered image with a slider to change the kernel size

# Add your code hered
cv.imshow('Median Filter', image)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
# GAUSSIAN BLUR FILTER

# Create a window for the trackbars

# Add your code here
cv.namedWindow('Gaussian Filter', cv.WINDOW_GUI_NORMAL)

# Create the trackbar

# Add your code here

# Callback function for the trackbar
def update_gaussian(val):
    # Get the actual values of the trackbars
    k = cv.getTrackbarPos("Kernel Size = ", "Gaussian Filter")
    s = cv.getTrackbarPos("Sigma = ", "Gaussian Filter")

    # Make sure the kernel size is odd
    if k % 2 == 0:
        k += 1
        
    # Call the filter
    gaussian(image, k, s*0.1)

cv.createTrackbar("Kernel Size = ", "Gaussian Filter", 0, 100, update_gaussian)
cv.createTrackbar("Sigma = ", "Gaussian Filter", 1, 100, update_gaussian)

# Show the filtered image with a slider to change the kernel size

# Add your code here
cv.imshow('Gaussian Filter', image)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
# BILATERAL FILTER

# Create a window for the trackbars

# Add your code here
cv.namedWindow('Bilateral Filter', cv.WINDOW_GUI_NORMAL)


# Create the trackbar

# Add your code here
# Callback function for the trackbar
def update_bilateral(val):
    # Get the actual values of the trackbars
    ss = cv.getTrackbarPos("S.Space = ", "Bilateral Filter")
    sr = cv.getTrackbarPos("S.Range = ", "Bilateral Filter")

    # Update the kernel size and make sure it is odd
    k = 6*ss
    if k % 2 == 0:
        k += 1
    
    # Call the filter
    bilateral.set_filter_size(k)
    bilateral(image, ss*0.1, sr*0.1)


cv.createTrackbar("S.Space = ", "Bilateral Filter", 1, 100, update_bilateral)
cv.createTrackbar("S.Range = ", "Bilateral Filter", 1, 100, update_bilateral)

# Show the filtered image with a slider to change the kernel size

# Add your code here
cv.imshow('Bilateral Filter', image)
cv.waitKey(0)
cv.destroyAllWindows()